# Deathstar Campaign Analysis

**Run:** `deathstar-20261026-1v1-10repeats`  
**Workload:** ribbon 1v1, 8 seeded matches, 1,801 aim candidates per turn, 10 repeats at 1/2/4/8 workers.

This is the throughput counterpart to `campaign_diagnostics.ipynb`. It analyzes the retrieved benchmark records rather than CPU or allocation profiles, which were not captured in this run. The simulator completed and archived all 40 trials; the enclosing remote job reports `failed` only because its final Python postflight validator requires `hashlib.file_digest`, absent from the host Python. The benchmark's own checks all passed.

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_ID = 'deathstar-20261026-1v1-10repeats'
repo_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
candidates = [
    repo_root / 'results' / 'benchmarks' / 'runs' / RUN_ID,
    repo_root / 'tmp' / 'deathstar-20261026' / RUN_ID,
]
run_dir = next((path for path in candidates if path.exists()), None)
if run_dir is None:
    raise FileNotFoundError('Download or place the run under results/benchmarks/runs or tmp/deathstar-20261026.')

trials = pd.read_csv(run_dir / 'trials.csv')
matches = pd.read_csv(run_dir / 'matches.csv')
manifest = json.loads((run_dir / 'manifest.json').read_text())
checks = json.loads((run_dir / 'checks.json').read_text())
archive_candidates = [
    repo_root / 'results' / 'benchmarks' / 'artifacts' / f'{RUN_ID}.tar.gz',
    repo_root / 'tmp' / 'deathstar-20261026' / f'{RUN_ID}.tar.gz',
]
archive = next((path for path in archive_candidates if path.exists()), archive_candidates[-1])

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'Run records: {run_dir}')
print(f'Archive: {archive}')

## 1. Integrity and completion

The same canonical gameplay history must result from every worker count and repeat. Completion and candidate totals are checked before interpreting timing data.

In [ ]:
all_checks_passed = all(check['status'] == 'passed' for check in checks['checks'])
archive_sha256 = hashlib.file_digest(archive.open('rb'), 'sha256').hexdigest() if archive.exists() else 'not available'
summary = {
    'benchmark_status': manifest['status'],
    'checks_passed': all_checks_passed,
    'trials': len(trials),
    'completed_matches': int(trials['completed_matches'].sum()),
    'candidate_evaluations': int(trials['candidates'].sum()),
    'unique_gameplay_hashes': trials['gameplay_sha256'].nunique(),
    'gameplay_sha256': trials['gameplay_sha256'].iloc[0],
    'archive_sha256': archive_sha256,
}
pd.Series(summary)

## 2. Worker scaling and repeat stability

Speedup and efficiency use the median one-worker campaign wall time as the baseline. The coefficient of variation (CV) describes repeat-to-repeat variability; it is not a confidence interval.

In [ ]:
scaling = trials.groupby('workers').agg(
    wall_median_s=('wall_seconds', 'median'), wall_min_s=('wall_seconds', 'min'),
    wall_max_s=('wall_seconds', 'max'), wall_mean_s=('wall_seconds', 'mean'),
    wall_std_s=('wall_seconds', 'std'), candidates_per_s=('candidates_per_second', 'median'),
    matches_per_hour=('completed_matches_per_hour', 'median'),
    mean_decision_s=('decision_mean_seconds', 'median'),
    p95_decision_s=('decision_p95_seconds', 'median'),
).reset_index()
baseline = scaling.loc[scaling.workers.eq(1), 'wall_median_s'].iloc[0]
scaling['speedup'] = baseline / scaling['wall_median_s']
scaling['efficiency_pct'] = 100 * scaling['speedup'] / scaling['workers']
scaling['cv_pct'] = 100 * scaling['wall_std_s'] / scaling['wall_mean_s']
scaling.round(3)

In [ ]:
fig, (wall_ax, speedup_ax) = plt.subplots(1, 2, figsize=(14, 5))
for workers, group in trials.groupby('workers'):
    wall_ax.scatter([workers] * len(group), group['wall_seconds'], color='#1f77b4', alpha=.75)
wall_ax.plot(scaling['workers'], scaling['wall_median_s'], '-o', color='#d62728', label='median')
wall_ax.set(xlabel='Workers', ylabel='Campaign wall time (s)', title='Complete campaign time by worker count')
wall_ax.legend()
speedup_ax.plot(scaling['workers'], scaling['workers'], '--', color='grey', label='ideal')
speedup_ax.plot(scaling['workers'], scaling['speedup'], '-o', color='#2ca02c', label='observed')
for row in scaling.itertuples():
    speedup_ax.annotate(f'{row.speedup:.2f}x\n{row.efficiency_pct:.0f}%', (row.workers, row.speedup),
                        xytext=(0, 9), textcoords='offset points', ha='center')
speedup_ax.set(xlabel='Workers', ylabel='Speedup vs. one worker', title='Scaling efficiency')
speedup_ax.set_xticks(scaling['workers'])
speedup_ax.legend()
plt.tight_layout()

## 3. Phase totals and critical-path effect

Phase durations are sums across worker tasks, not elapsed campaign time. At higher worker counts the summed simulation time therefore exceeds wall time; the ratio below visualizes useful concurrent work and should not be read as a percentage breakdown.

In [ ]:
phase_columns = ['sim_phase_seconds', 'candidate_reset_phase_seconds', 'selection_phase_seconds',
                 'actual_shot_phase_seconds', 'checkpoint_phase_seconds']
phases = trials.groupby('workers')[phase_columns + ['wall_seconds']].median().reset_index()
phases['sim_to_wall'] = phases['sim_phase_seconds'] / phases['wall_seconds']
display(phases.round(4))

ax = phases.set_index('workers')[phase_columns].plot(kind='bar', stacked=True, figsize=(11, 5), colormap='tab20')
ax.set(xlabel='Workers', ylabel='Summed median phase time (s)', title='Median phase totals across the eight matches')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()

## 4. Match balance and memory

The per-match chart uses the one-worker trials to expose intrinsic match cost without scheduling overlap. RSS is cumulative process RSS and sampled at process boundaries; it is not a live-heap measurement.

In [ ]:
one_worker_matches = matches[matches.workers.eq(1)]
match_summary = one_worker_matches.groupby(['match_id', 'turns', 'termination_reason'], as_index=False).agg(
    candidates=('candidates', 'first'), median_wall_s=('wall_seconds', 'median'),
    winner=('winner', 'first'), player1_score=('player1_score', 'first'), player2_score=('player2_score', 'first'))
display(match_summary.sort_values('median_wall_s', ascending=False).round(3))

fig, (match_ax, rss_ax) = plt.subplots(1, 2, figsize=(14, 5))
plot_matches = match_summary.sort_values('median_wall_s')
match_ax.barh(plot_matches.match_id.astype(str), plot_matches.median_wall_s, color='#9467bd')
match_ax.set(xlabel='Median one-worker match wall time (s)', ylabel='Match', title='Intrinsic match-cost imbalance')
rss = trials.groupby('workers').agg(rss_mib=('process_rss_bytes', lambda x: x.median()/2**20),
                                        peak_rss_mib=('process_peak_rss_bytes', lambda x: x.max()/2**20)).reset_index()
rss_ax.plot(rss.workers, rss.rss_mib, '-o', label='post-output RSS')
rss_ax.plot(rss.workers, rss.peak_rss_mib, '-o', label='cumulative process peak')
rss_ax.set(xlabel='Workers', ylabel='MiB', title='Process RSS by worker count')
rss_ax.set_xticks(rss.workers)
rss_ax.legend()
plt.tight_layout()

## Interpretation

The valid benchmark result is 150.324 s at one worker and 28.741 s at eight workers (5.23x speedup, 65.4% worker efficiency). Two workers retain 98.8% efficiency; degradation becomes material at four and eight workers, consistent with only eight unevenly expensive matches and the critical path of the slowest assigned work. The ten-repeat spread is low (CV 0.27%–1.13%).

This notebook establishes throughput, scaling, phase totals, deterministic gameplay, and boundary RSS only. It cannot attribute CPU time or allocations; use a separate `campaign-diagnostics/1` run on Deathstar for those questions. Before treating the remote job as fully postflight-clean, repair the Python validator compatibility issue and rerun that validator against the retained archive.